In [1]:
#CNN classification Project
import torchvision
import torch
from torchvision import transforms
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

In [2]:
#CIFAR-10 dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.backends.cudnn.benchmark = True

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2023,0.1994,0.2010))
])

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2023,0.1994,0.2010))
])

train_dataset = torchvision.datasets.CIFAR10(root="./data",train=True,download=True,transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root="./data",train=False,download=False, transform = transform_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers = 2,
                          pin_memory=True, persistent_workers = True)

test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False,
                         pin_memory=True)

#CNN

class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.Conv2d1 = nn.Conv2d(3,32,3, padding = 1)
    self.BatchNorm1 = nn.BatchNorm2d(32)
    self.Conv2d2 = nn.Conv2d(32,64,3, padding = 1)
    self.BatchNorm2 = nn.BatchNorm2d(64)
    self.Conv2d3 = nn.Conv2d(64,128,3, padding = 1)
    self.BatchNorm3 = nn.BatchNorm2d(128)

    self.Dropout = nn.Dropout(0.2)
    self.nnMaxPool2d = nn.MaxPool2d(2,2)
    self.nnLinear = nn.Linear(2048,10)

  def forward(self,x):
    x = (self.Conv2d1(x))
    x = F.relu(self.BatchNorm1(x))
    x = (self.nnMaxPool2d(x))
    x = (self.Conv2d2(x))
    x = F.relu(self.BatchNorm2(x))
    x = (self.nnMaxPool2d(x))
    x = (self.Conv2d3(x))
    x = F.relu(self.BatchNorm3(x))
    x = (self.nnMaxPool2d(x))
    x = x.view(x.size(0), -1)
    x = self.Dropout(x)
    output = self.nnLinear(x)
    return output

class Trainer:
  def __init__(self, model, TrainSet, TestSet):
    self.model = model
    self.TrainSet = TrainSet
    self.TestSet = TestSet
    self.criterion = nn.CrossEntropyLoss()
    self.optimizer = optim.Adam(model.parameters(), lr=0.001)

  def train(self, epochs):
    for epoch in range(epochs):

      print(f'Epoch: {epoch+1} started', flush = True)

      for inputs, targets in train_loader:
        inputs, targets = inputs.to(device, non_blocking = True), targets.to(device, non_blocking = True)
        self.optimizer.zero_grad()
        outputs = self.model(inputs)
        loss = self.criterion(outputs, targets)
        loss.backward()
        self.optimizer.step()

      print(f'Epoch: {epoch+1} completed', flush = True)

  def evaluate(self):
   self.model.eval()
   all_preds = []
   all_targets = []

   with torch.no_grad():
      for inputs, targets in test_loader:
          inputs, targets = inputs.to(device, non_blocking = True), targets.to(device, non_blocking = True)
          outputs = self.model(inputs)
          _, preds = torch.max(outputs, 1)
          all_preds.append(preds)
          all_targets.append(targets)

   all_preds = torch.cat(all_preds)
   all_targets = torch.cat(all_targets)

   accuracy = (all_preds == all_targets).float().mean()

   all_preds = all_preds.cpu().numpy()
   all_targets = all_targets.cpu().numpy()

   conf_matrix = confusion_matrix(all_targets, all_preds)

   report = classification_report(all_targets, all_preds)

   return print(f'Accuracy: {round(accuracy.item(),3)}'), print(f'Confusion Matrix: \n {conf_matrix}'), print(f'Report: {report}')


cnn = CNN()
cnn = cnn.to(device)


100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s]


In [3]:
my_trainer = Trainer(cnn, train_loader, test_loader)

my_trainer.train(epochs = 30)
my_trainer.evaluate()


#without batch normalization (and 10 epochs):
#0.67 accuracy
# "easiest" class to predict correctly: 1 or 8
# "most difficult" class to predict correctly: 3

#with batch normalization (and 10 epochs):
# 0.66 accuracy
# Increased performance in Precision, but worse results in Recall
# Significantly downgrade for Class 3


#Increasing epochs to 15:
# 0.68 accuracy
# Improved in class 3 and got a more bilanced result

#Adding Dropout and data Augmentation (+ increasing to 25 epochs):
# 0.69 accuracy
# improved in every class, but worse result in class 3

#Adding cudnn.benchmark (because input shape is static) and a third Convolutional layer
# ( + increasing to 30 epochs):
# 0.8 accuracy
# Great improvement in every class, even in class 3 that reached a decent precision


Epoch: 1 started
Epoch: 1 completed
Epoch: 2 started
Epoch: 2 completed
Epoch: 3 started
Epoch: 3 completed
Epoch: 4 started
Epoch: 4 completed
Epoch: 5 started
Epoch: 5 completed
Epoch: 6 started
Epoch: 6 completed
Epoch: 7 started
Epoch: 7 completed
Epoch: 8 started
Epoch: 8 completed
Epoch: 9 started
Epoch: 9 completed
Epoch: 10 started
Epoch: 10 completed
Epoch: 11 started
Epoch: 11 completed
Epoch: 12 started
Epoch: 12 completed
Epoch: 13 started
Epoch: 13 completed
Epoch: 14 started
Epoch: 14 completed
Epoch: 15 started
Epoch: 15 completed
Epoch: 16 started
Epoch: 16 completed
Epoch: 17 started
Epoch: 17 completed
Epoch: 18 started
Epoch: 18 completed
Epoch: 19 started
Epoch: 19 completed
Epoch: 20 started
Epoch: 20 completed
Epoch: 21 started
Epoch: 21 completed
Epoch: 22 started
Epoch: 22 completed
Epoch: 23 started
Epoch: 23 completed
Epoch: 24 started
Epoch: 24 completed
Epoch: 25 started
Epoch: 25 completed
Epoch: 26 started
Epoch: 26 completed
Epoch: 27 started
Epoch: 27 co

(None, None, None)